In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load our processed parquet file
df = pd.read_parquet('../data/processed/iot_sentinel.parquet')

# Compute Inter-Arrival Time (IAT) per flow
df = df.sort_values(by=['flow_id', 'time'])
df['iat'] = df.groupby('flow_id')['time'].diff().fillna(0)

In [3]:
def process_iot_data(df):
    """
    Processes raw IoT packet data into a behavioral dataset.
    1. Uses the pre-extracted device_name.
    2. Groups packets into flows using the flow_id.
    3. Calculates statistical features (Mean, Std, etc.).
    """
    
    # Drop rows that don't match our known IoT devices just in case
    df = df.dropna(subset=['device_name'])

    # --- Behavioral Aggregation ---
    behavioral_df = df.groupby('flow_id').agg({
        # Packet Length Stats (Column was named 'length' in our script)
        'length': ['mean', 'std', 'max', 'min', 'sum'],
        
        # Temporal Stats (Inter-Arrival Time)
        'iat': ['mean', 'std', 'max'],
        
        # Protocol Info (e.g., most common protocol in the flow)
        'protocol': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
        
        # Keep the label (it's the same for the whole flow)
        'device_name': 'first'
    })

    # Flatten the multi-index columns (e.g., 'length_mean')
    behavioral_df.columns = ['_'.join(col).strip() for col in behavioral_df.columns.values]
    behavioral_df = behavioral_df.rename(columns={'device_name_first': 'target_label'})

    # --- Anti-Overfitting Drop ---
    behavioral_df = behavioral_df.fillna(0)

    return behavioral_df

In [4]:
# Run the processing function and display the results
behavioral_df = process_iot_data(df)
behavioral_df.head()

,length_mean,length_std,length_max,length_min,length_sum,iat_mean,iat_std,iat_max,protocol_<lambda>,target_label
flow_id,,,,,,,,,,
0000e6c20323a1fb7f59da389f1c27d5,166.272727,261.542383,934,66,1829,0.001779,0.001991,0.005550,6.0,HueSwitch
0001f6083d630a6be0d039c64b761ded,56.914286,3.042555,60,54,1992,2.157110,2.480101,5.011431,6.0,D-LinkDoorSensor
0003f4055b64d2f70dc3f8a495925a7b,146.000000,197.135994,713,66,1606,0.001553,0.001601,0.005139,6.0,HueSwitch
000445e901942c1204a9c1dd2ebea641,74.000000,0.000000,74,74,296,1.754420,1.709725,4.005742,6.0,WeMoLink
0005b83ec56004ac0659b6fea663fdb2,74.000000,0.000000,74,74,74,0.000000,0.000000,0.000000,6.0,WeMoInsightSwitch2
